In [1]:
import numpy as np
from sklearn.datasets import make_moons

# ---- 1. Data Setup ----
X, y = make_moons(n_samples=400, noise=0.20, random_state=1)
y = y.reshape(-1, 1).astype(float)
X = (X - X.mean(0)) / X.std(0) # Standardize

# ---- 2. Helpers & Initialization ----
def relu(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def sig(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500))) # Clipped for numerical stability

def init(sizes, seed=0):
    rng = np.random.default_rng(seed)
    P = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        W = rng.normal(0, np.sqrt(2 / a), size=(a, b)) # He initialization
        b_vec = np.zeros((1, b))
        P.append([W, b_vec])
    return P

# ---- 3. Forward + Backward Pass ----
def grads(P, Xb, yb):
    zs, acts, a = [], [Xb], Xb

    # Forward Pass
    for i, (W, b) in enumerate(P):
        z = a @ W + b
        zs.append(z)
        a = sig(z) if i == len(P) - 1 else relu(z)
        acts.append(a)

    m = Xb.shape[0]
    g = [None] * len(P)

    # dL/dz for Sigmoid + Binary Cross-Entropy
    dz = (acts[-1] - yb) / m

    # Backward Pass
    for i in reversed(range(len(P))):
        dW = acts[i].T @ dz
        db = np.sum(dz, axis=0, keepdims=True)
        g[i] = [dW, db]

        if i > 0:
            dz = (dz @ P[i][0].T) * drelu(zs[i - 1])

    return g

# ---- 4. Evaluation Helper ----
def accuracy(P, X, y):
    a = X
    for i, (W, b) in enumerate(P):
        a = sig(a @ W + b) if i == len(P) - 1 else relu(a @ W + b)
    return np.mean((a > 0.5) == (y > 0.5))

# ---- 5. Hyperparameters ----
sizes = [2, 16, 16, 1]
EPOCHS = 200

# ---- (A) Batch Gradient Descent ----
lr_bgd = 0.5
P = init(sizes, seed=0)

for epoch in range(EPOCHS):
    g = grads(P, X, y)
    for k in range(len(P)):
        P[k][0] -= lr_bgd * g[k][0]
        P[k][1] -= lr_bgd * g[k][1]

print("Batch GD accuracy :", accuracy(P, X, y))

# ---- (B) Stochastic Gradient Descent (Mini-Batch) ----
lr_sgd = 0.1  # Adjusted for mini-batch stability
P = init(sizes, seed=0)
B = 16
idx = np.arange(len(X))
rng = np.random.default_rng(0)

for epoch in range(EPOCHS):
    rng.shuffle(idx)
    for s in range(0, len(X), B):
        b = idx[s : s + B]
        g = grads(P, X[b], y[b])
        for k in range(len(P)):
            P[k][0] -= lr_sgd * g[k][0]
            P[k][1] -= lr_sgd * g[k][1]

print("SGD accuracy      :", accuracy(P, X, y))

Batch GD accuracy : 0.965
SGD accuracy      : 0.96


In [2]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_moons
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD

# ---- Data Preparation ----
X, y = make_moons(n_samples=400, noise=0.20, random_state=1)
y = y.reshape(-1, 1).astype(float)
X = (X - X.mean(0)) / X.std(0)  # standardize

# Helper to build a fresh, un-trained model architecture
def build_model(learning_rate):
    model = Sequential([
        Input(shape=(2,)),
        Dense(16, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        loss='binary_crossentropy',
        optimizer=SGD(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

# ---- 1. Batch Gradient Descent (Full dataset in 1 batch) ----
tf.random.set_seed(0)
model_bgd = build_model(learning_rate=0.5)
model_bgd.fit(X, y, epochs=200, batch_size=len(X), verbose=0)
loss_bgd, acc_bgd = model_bgd.evaluate(X, y, verbose=0)
print(f"Batch GD Accuracy : {acc_bgd:.4f}")

# ---- 2. Mini-Batch SGD (Batch size = 16) ----
tf.random.set_seed(0)
model_sgd = build_model(learning_rate=0.1)  # Lower learning rate for mini-batches
model_sgd.fit(X, y, epochs=200, batch_size=16, verbose=0)
loss_sgd, acc_sgd = model_sgd.evaluate(X, y, verbose=0)
print(f"SGD Accuracy      : {acc_sgd:.4f}")

Batch GD Accuracy : 0.9625
SGD Accuracy      : 0.9650
